In [119]:
import pandas as pd
import zipfile
import os

In [120]:
# Histórico 2010 a sep 2025
!wget -O metro_historico.zip https://raw.githubusercontent.com/AndrsGzRo/prediccion-afluencia-metrocdmx/refs/heads/main/Data/raw/Afluencia_Diaria_MetroCDMX.zip

# Octubre a Diciembre
!wget -O afluencia_q4_2025.csv https://raw.githubusercontent.com/AndrsGzRo/prediccion-afluencia-metrocdmx/refs/heads/main/Data/raw/data-2026-01-28.csv


--2026-02-10 19:08:20--  https://raw.githubusercontent.com/AndrsGzRo/prediccion-afluencia-metrocdmx/refs/heads/main/Data/raw/Afluencia_Diaria_MetroCDMX.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7229852 (6.9M) [application/zip]
Saving to: ‘metro_historico.zip’

metro_historico.zip 100%[===================>]   6.89M  --.-KB/s    in 0.09s   

2026-02-10 19:08:20 (75.7 MB/s) - ‘metro_historico.zip’ saved [7229852/7229852]

--2026-02-10 19:08:20--  https://raw.githubusercontent.com/AndrsGzRo/prediccion-afluencia-metrocdmx/refs/heads/main/Data/raw/data-2026-01-28.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|

In [121]:
dfs = []
with zipfile.ZipFile('metro_historico.zip', 'r') as z:
    for file in z.namelist():
        if file.endswith('.csv'):
            df = pd.read_csv(z.open(file))
            dfs.append(df)
df_hist = pd.concat(dfs, ignore_index=True)

In [122]:
df_q4_2025 = pd.read_csv("afluencia_q4_2025.csv")

In [123]:
df_q4_2025

,fecha,mes,anio,linea,estacion,tipo_pago,afluencia,temporal_fecha,..anio_fecha
0,2025-10-01,Octubre,2025,Linea 1,PantitlÃ¡n,Boleto,0,2025-10,2025
1,2025-10-01,Octubre,2025,Linea 1,PantitlÃ¡n,Prepago,33468,2025-10,2025
2,2025-10-01,Octubre,2025,Linea 1,PantitlÃ¡n,Gratuidad,5673,2025-10,2025
3,2025-10-01,Octubre,2025,Linea 1,Zaragoza,Boleto,0,2025-10,2025
4,2025-10-01,Octubre,2025,Linea 1,Zaragoza,Prepago,21151,2025-10,2025
...,...,...,...,...,...,...,...,...,...
53815,2025-12-31,Diciembre,2025,Linea 12,Insurgentes Sur,Prepago,15645,2025-12,2025
53816,2025-12-31,Diciembre,2025,Linea 12,Insurgentes Sur,Gratuidad,1587,2025-12,2025
53817,2025-12-31,Diciembre,2025,Linea 12,Mixcoac,Boleto,0,2025-12,2025
53818,2025-12-31,Diciembre,2025,Linea 12,Mixcoac,Prepago,6825,2025-12,2025


In [124]:
df_q4_2025 = df_q4_2025.copy()
df_q4_2025 = (
    df_q4_2025
    .groupby(['fecha','anio','mes','linea','estacion'])
    ['afluencia']
    .sum()
    .reset_index()
)

In [125]:
def limpiar_estaciones(df):
    '''Función para limpiar,corregir y reemplazar el nombre de las
    estaciones del Metro de la Ciudad de México'''

    MAPA_ESTACIONES = {
    'Isabel la CatÃ³lica':'Isabel la Católica', 'Pino SuÃ¡rez':'Pino Suárez',
    'GÃ³mez FarÃ\xadas':'Gómez Farías','La Villa/BasÃ\xadlica':'La Villa-Basílica',
    'PantitlÃ¡n':'Pantitlán','VelÃ³dromo':'Velódromo','RefinerÃ\xada':'Refinería',
    'EtiopÃ\xada/Plaza de la Transparencia':'Etiopía/ Plaza de la Transparencia',
    'DivisiÃ³n del Norte':'División del Norte',
    'FerrerÃ\xada/Arena Ciudad de MÃ©xico':'Ferretería/Arena Ciudad de México',
    'Instituto del PetrÃ³leo':'Instituto del Petróleo','San Juan de LetrÃ¡n':'San Juan Letrán',
    'OceanÃ\xada':'Ocenía','NezahualcÃ³yotl':'Nezahualcóyotl','RevoluciÃ³n':'Revolución',
    'JuÃ¡rez':'Juárez','OlÃ\xadmpica':'Olímpica','CulhuacÃ¡n':'Culhuacán',
    'PeÃ±Ã³n Viejo':'Peñón Viejo','San AndrÃ©s TomatlÃ¡n':'San Andrés Tomatlán',
    'AragÃ³n':'Aragón','MartÃ\xadn Carrera':'Martín Carrera','Aquiles SerdÃ¡n':'Aquiles Serdán',
    'San JoaquÃ\xadn':'San Joaquín','TezozÃ³moc':'Tezozómoc','CuitlÃ¡huac':'Cuitláhuac',
    'Terminal AÃ©rea':'Terminal Aérea','Centro MÃ©dico':'Centro Médico','CoyoacÃ¡n':'Coyoacán',
    'ZÃ³calo/Tenochtitlan':'Zócalo/Tenochtitlan','Boulevard Puerto AÃ©reo':'Boulevard Puerto Aéreo',
    'Villa de CortÃ©s':'Villa de Cortés','TalismÃ¡n':'Talismán','Valle GÃ³mez':'Valle Gómez',
    'PolitÃ©cnico':'Politécnico','NiÃ±os HÃ©roes':'Niños Héroes','TasqueÃ±a':'Tasqueña',
    'JuanacatlÃ¡n':'Juanacatlán','Plaza AragÃ³n':'Plaza Aragón','ZapotitlÃ¡n':'Zapotitlán',
    'RÃ\xado de los Remedios':'Río de los Remedios','MÃºzquiz':'Múzquiz',
    'EscuadrÃ³n 201':'Escuadrón 201','PerifÃ©rico Oriente':'Periférico Oriente',
    'TlÃ¡huac':'Tláhuac','San LÃ¡zaro':'San Lázaro','Villa de AragÃ³n':'Villa de Aragón',
    'AgrÃ\xadcola Oriental':'Agrícola Oriental','Deportivo OceanÃ\xada':'Deportivo Oceanía',
    'Bosque de AragÃ³n':'Bosque de Aragón','ConstituciÃ³n de 1917':'Constitución de 1917',
    'CuauhtÃ©moc':'Cuauhtémoc','Ricardo Flores MagÃ³n':'Ricardo Flores Magón',
    'LÃ¡zaro CÃ¡rdenas':'Lázaro Cárdenas','GÃ³mez Farias':'Gómez Farías',
    'PeÃ±Ã³n viejo':'Peñón Viejo','Miguel Ã\x81ngel de Quevedo':'Miguel Ángel de Quevedo'
    }
    df['estacion'] = df['estacion'].replace(MAPA_ESTACIONES)
    return df

# Aplicando función a los DataFrames
df_hist = limpiar_estaciones(df_hist)
df_q4_2025 = limpiar_estaciones(df_q4_2025)

In [126]:
def limpiar_lineas(df):
    '''Función para limpiar,corregir y reemplazar el nombre de las
    líneas del Metro de la Ciudad de México'''
    MAPA_LINEAS = {
    'LÃ\xadnea 1': 'Línea 1','LÃ\xadnea 2': 'Línea 2','LÃ\xadnea 3': 'Línea 3',
    'LÃ\xadnea 4': 'Línea 4','LÃ\xadnea 5': 'Línea 5','LÃ\xadnea 6': 'Línea 6',
    'LÃ\xadnea 7': 'Línea 7','LÃ\xadnea 8': 'Línea 8','LÃ\xadnea 9': 'Línea 9',
    'LÃ\xadnea 12': 'Línea 12','LÃ\xadnea A': 'Línea A','LÃ\xadnea B': 'Línea B',
    'Linea 1':'Línea 1', 'Linea 6':'Línea 6','Linea 9':'Línea 9','Linea 8':'Línea 8',
    'Linea 5':'Línea 5','Linea 7':'Línea 7','Linea 3':'Línea 3','Linea 4':'Línea 4',
    'Linea 2':'Línea 2','Linea B':'Línea B','Linea 12':'Línea 12','Linea A':'Línea A'
    }
    df['linea'] = df['linea'].replace(MAPA_LINEAS)
    return df

# Aplicando función a los DataFrames
df_hist = limpiar_lineas(df_hist)
df_q4_2025 = limpiar_lineas(df_q4_2025)

In [127]:
MAPA_MESES = {
    'enero': 1,
    'febrero': 2,
    'marzo': 3,
    'abril': 4,
    'mayo': 5,
    'junio': 6,
    'julio': 7,
    'agosto': 8,
    'septiembre': 9,
    'setiembre': 9,
    'octubre': 10,
    'noviembre': 11,
    'diciembre': 12
}

MAPA_MESES_INV = {
    1: 'enero', 2: 'febrero', 3: 'marzo', 4: 'abril',
    5: 'mayo', 6: 'junio', 7: 'julio', 8: 'agosto',
    9: 'septiembre', 10: 'octubre', 11: 'noviembre', 12: 'diciembre'
}

In [128]:
def normalizar_df_metro(df: pd.DataFrame) -> pd.DataFrame:
    '''Función para normalizar un DataFrame de la Afluencia del Metro de la CDMX'''
    df = df.copy()

    # Normalizar nombres
    df.columns = (
        df.columns
        .str.lower()
        .str.strip()
        .str.replace(" ","_")
    )

    # Esquema oficial
    columnas_base = [
    'fecha','anio','mes','linea','estacion','afluencia'
    ]

    df = df[[c for c in columnas_base if c in df.columns]]

    # Fecha y año
    df['fecha'] = pd.to_datetime(df['fecha'])
    df['anio'] = df['anio'].astype(int)

    # Mes
    mes_raw = (
        df['mes']
        .astype(str)
        .str.lower()
        .str.strip()
    )
    df['mes_num'] = mes_raw.map(MAPA_MESES).astype(int)
    df['mes_nom'] = df['mes_num'].map(MAPA_MESES_INV)

    # Eliminar mes original
    df = df.drop(columns=['mes'])

    # Linea
    df['linea'] = df['linea'].astype(str)

    # Afluencia
    df['afluencia'] = df['afluencia'].astype(int)
    return df

# Aplicando funciones
df_hist = normalizar_df_metro(df_hist)
df_q4_2025 = normalizar_df_metro(df_q4_2025)

In [130]:
# Concat
df_full = pd.concat(
    [df_hist,df_q4_2025],
    ignore_index=True
)

df_full = df_full.drop_duplicates(
    subset = ['fecha','linea','estacion']
)

In [131]:
# Validación
df_full.shape
df_full['fecha'].min(), df_full['fecha'].max()


(Timestamp('2010-01-01 00:00:00'), Timestamp('2025-12-31 00:00:00'))

In [132]:
df_full['anio'].value_counts().sort_index()


,count
anio,
2010,71175
2011,71175
2012,71370
2013,71175
2014,71175
2015,71175
2016,71370
2017,71175
2018,71175
